# 5단계: End-to-end 검증 (Colab, GPU)

4단계에서 고른 top-k 문장을 근거로 넣어 SUPPORTED / NOT_SUPPORTED를 판정한다. 베이스라인의 최종 점수다.

| 비교 대상 | test macro-F1 |
|---|---|
| 1단계 claim-only (하한선) | 0.620 |
| 2단계 oracle (상한선) | 0.800 |

**검증기 학습 데이터** (`TRAIN_ON`)
- `'retrieved'` (기본, 원본 HoVer 방식): train claim에도 4단계가 고른 top-k 문장을 넣어 학습. 평가 때와 같은 형태의 입력(잡음 섞인 근거)으로 배운다
- `'gold'`: 2단계처럼 정답 근거로 학습. 깨끗한 근거만 보고 배운 모델이 잡음 섞인 근거에서 얼마나 버티는지 볼 수 있다

주의: train의 top-k 문장은 그 train claim으로 학습한 문장 선택 모델이 고른 것이라 val/test보다 검색이 잘 되어 있다. 원본도 같은 방식이다.

**지표**
- accuracy, macro-F1
- **HoVer score**: 라벨이 맞고 **정답 근거 문장이 top-k 안에 모두** 있을 때만 1점. 근거 없이 우연히 맞힌 경우를 걸러 낸다

| 셀 | 역할 |
|---|---|
| 1 | 0단계 데이터 + 4단계 문장 선택 결과 불러오기 |
| 2 | 지표 함수 |
| 3 | 토크나이즈 |
| 4 | 검증기 학습 (val은 검색된 근거로 평가해 최고 모델 선택) |
| 5 | test 평가: 검색된 근거 vs 정답 근거, val에서 k 비교 |
| 6 | 예측 저장 (오류 분석용) |

In [10]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers datasets accelerate

import tqdm.auto, tqdm.std
tqdm.auto.tqdm = tqdm.std.tqdm  # VSCode+Colab에서는 위젯 진행 막대가 안 그려져서 텍스트 막대로 바꾼다 (transformers/datasets import 전에)

import unicodedata
import numpy as np, pandas as pd
nfc = lambda s: unicodedata.normalize('NFC', s)  # 정답 제목은 NFC, 위키 DB/TF-IDF 제목은 NFD라서 비교 전에 통일
PROC = '/content/drive/MyDrive/fact-verification/data/processed'
LABELS = ['NOT_SUPPORTED', 'SUPPORTED']

TAG = '_mh'  # 베이스라인: '' / 08 multi-hop 문서 검색 결과를 쓰려면 '_mh'
dfs = {}
for n in ['train', 'val', 'test']:
    d = pd.read_json(f'{PROC}/hover2_{n}.jsonl', lines=True)
    r = pd.read_json(f'{PROC}/hover2_{n}_sentret{TAG}.jsonl', lines=True)
    d = d.merge(r, on='uid', validate='1:1')
    d['labels'] = d.label.map(LABELS.index)
    d['gold_sents'] = d.supporting_facts.map(lambda sf: {(nfc(t), i) for t, i in sf})
    d['gold_context'] = d.evidence.map(lambda ev: ' '.join(s for s in ev if s))
    dfs[n] = d
{n: len(d) for n, d in dfs.items()}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


{'train': 8166, 'val': 886, 'test': 1126}

In [11]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

K_SENT = 5  # 4단계 val 결과로 고른 값

def retrieved_context(pred, k):
    return ' '.join(s for _, _, s, _ in pred[:k])  # pred 원소: [제목, 문장 번호, 문장, 점수]

def evidence_found(d, k):
    return np.array([gold <= {(nfc(t), int(i)) for t, i, *_ in pred[:k]} for gold, pred in zip(d.gold_sents, d.pred_sents)])

def scores(y, pred):
    return {'acc': round(accuracy_score(y, pred), 3), 'macro_f1': round(f1_score(y, pred, average='macro'), 3)}

In [12]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = 'roberta-base'
MAX_LEN = 384  # 문장 5~10개 + claim
TRAIN_ON = 'retrieved'  # 'retrieved' | 'gold'
tok = AutoTokenizer.from_pretrained(MODEL)

def make_ds(d, context):
    return Dataset.from_dict({'claim': d.claim.tolist(), 'context': list(context), 'labels': d.labels.tolist()}) \
        .map(lambda b: tok(b['claim'], b['context'], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=['claim', 'context'])

def retrieved_ds(n, k=K_SENT):
    return make_ds(dfs[n], dfs[n].pred_sents.map(lambda p: retrieved_context(p, k)))

train_ds = retrieved_ds('train') if TRAIN_ON == 'retrieved' else make_ds(dfs['train'], dfs['train'].gold_context)
val_ds = retrieved_ds('val')  # 모델 선택도 실제 상황(검색된 근거)으로 한다

Map: 100%|██████████| 886/886 [00:00<00:00, 1894.43 examples/s]


In [13]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

counts = np.bincount(dfs['train'].labels)
weight = torch.tensor(len(dfs['train']) / (2 * counts), dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=weight.to(out.logits.device))
        return (loss, out) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(
    output_dir=f'/content/out/verify_{TRAIN_ON}',
    num_train_epochs=3, learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=64, logging_steps=50, disable_tqdm=True,  # 진행 표 대신 손실을 50 step마다 텍스트로 출력
    warmup_steps=0.1, weight_decay=0.01, fp16=True, seed=42,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='macro_f1', report_to='none',
)
trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok),
                          compute_metrics=lambda p: scores(p.label_ids, p.predictions.argmax(-1)))
trainer.train()
# Training Loss가 0.693(ln 2)에 계속 머물면 학습이 무너진 것이다. seed를 바꿔 다시 돌린다

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 843.83it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.7053', 'grad_norm': '1.942', 'learning_rate': '1.273e-05', 'epoch': '0.1953'}
{'loss': '0.6977', 'grad_norm': '1.144', 'learning_rate': '1.936e-05', 'epoch': '0.3906'}
{'loss': '0.692', 'grad_norm': '3.039', 'learning_rate': '1.792e-05', 'epoch': '0.5859'}
{'loss': '0.6957', 'grad_norm': '1.28', 'learning_rate': '1.647e-05', 'epoch': '0.7812'}
{'loss': '0.6915', 'grad_norm': '0.6586', 'learning_rate': '1.502e-05', 'epoch': '0.9766'}
{'eval_loss': '0.6882', 'eval_acc': '0.683', 'eval_macro_f1': '0.569', 'eval_runtime': '3.886', 'eval_samples_per_second': '228', 'eval_steps_per_second': '3.603', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.51s/it]


{'loss': '0.6848', 'grad_norm': '5.863', 'learning_rate': '1.357e-05', 'epoch': '1.172'}
{'loss': '0.6652', 'grad_norm': '3.828', 'learning_rate': '1.213e-05', 'epoch': '1.367'}
{'loss': '0.5888', 'grad_norm': '11.5', 'learning_rate': '1.068e-05', 'epoch': '1.562'}
{'loss': '0.5252', 'grad_norm': '13.26', 'learning_rate': '9.233e-06', 'epoch': '1.758'}
{'loss': '0.5125', 'grad_norm': '14.09', 'learning_rate': '7.786e-06', 'epoch': '1.953'}
{'eval_loss': '0.492', 'eval_acc': '0.764', 'eval_macro_f1': '0.744', 'eval_runtime': '3.901', 'eval_samples_per_second': '227.1', 'eval_steps_per_second': '3.589', 'epoch': '2'}


Writing model shards: 100%|██████████| 1/1 [00:16<00:00, 16.62s/it]


{'loss': '0.4101', 'grad_norm': '9.95', 'learning_rate': '6.339e-06', 'epoch': '2.148'}
{'loss': '0.4791', 'grad_norm': '10.21', 'learning_rate': '4.891e-06', 'epoch': '2.344'}
{'loss': '0.4288', 'grad_norm': '9.454', 'learning_rate': '3.444e-06', 'epoch': '2.539'}
{'loss': '0.4564', 'grad_norm': '15.3', 'learning_rate': '1.997e-06', 'epoch': '2.734'}
{'loss': '0.4026', 'grad_norm': '8.103', 'learning_rate': '5.499e-07', 'epoch': '2.93'}
{'eval_loss': '0.4459', 'eval_acc': '0.828', 'eval_macro_f1': '0.798', 'eval_runtime': '3.883', 'eval_samples_per_second': '228.2', 'eval_steps_per_second': '3.606', 'epoch': '3'}


Writing model shards: 100%|██████████| 1/1 [00:20<00:00, 20.51s/it]


{'train_runtime': '517.7', 'train_samples_per_second': '47.33', 'train_steps_per_second': '1.484', 'train_loss': '0.5716', 'epoch': '3'}


TrainOutput(global_step=768, training_loss=0.5715797537316879, metrics={'train_runtime': 517.6599, 'train_samples_per_second': 47.325, 'train_steps_per_second': 1.484, 'train_loss': 0.5715797537316879, 'epoch': 3.0})

In [14]:
# 웹 데모(app.py)에서 쓰도록 학습한 모델을 Drive에 저장한다
SAVE = '/content/drive/MyDrive/fact-verification/models/verifier'
trainer.save_model(SAVE); tok.save_pretrained(SAVE)
!ls -lh {SAVE}

Writing model shards: 100%|██████████| 1/1 [00:12<00:00, 12.29s/it]


total 479M
-rw------- 1 root root  691 Oct 27 05:05 config.json
-rw------- 1 root root 476M Oct 27 05:05 model.safetensors
-rw------- 1 root root  388 Oct 27 05:05 tokenizer_config.json
-rw------- 1 root root 3.4M Oct 27 05:05 tokenizer.json
-rw------- 1 root root 5.1K Oct 27 05:05 training_args.bin


In [15]:
def predict(ds):
    return trainer.predict(ds).predictions.argmax(-1)

# val에서 k 비교 (재학습 없이 넣는 문장 수만 바꾼다)
y = dfs['val'].labels.values
for k in [3, 5, 7, 10]:
    pred = predict(retrieved_ds('val', k))
    print(f'val k={k:2d}', scores(y, pred), 'HoVer', round(float(np.mean((pred == y) & evidence_found(dfs['val'], k))), 3))

# test: 검색된 근거(최종 점수) vs 정답 근거(같은 검증기의 상한)
d = dfs['test']; y = d.labels.values
pred = predict(retrieved_ds('test'))
found = evidence_found(d, K_SENT)
print(f'\ntest 검색된 근거 k={K_SENT}', scores(y, pred), 'HoVer', round(float(np.mean((pred == y) & found)), 3))
print(pd.DataFrame(confusion_matrix(y, pred), index=[f'true_{l}' for l in LABELS], columns=[f'pred_{l}' for l in LABELS]))
for name, m in [('근거를 다 찾은 claim', found), ('근거를 놓친 claim   ', ~found)]:
    if m.any():
        print(f'  {name} {m.sum():4d}', scores(y[m], pred[m]))
gold_pred = predict(make_ds(d, d.gold_context))
print('test 정답 근거          ', scores(y, gold_pred))
d['pred'] = [LABELS[i] for i in pred]

Map: 100%|██████████| 886/886 [00:00<00:00, 2201.02 examples/s]


val k= 3 {'acc': 0.769, 'macro_f1': 0.747} HoVer 0.503


Map: 100%|██████████| 886/886 [00:00<00:00, 1951.93 examples/s]


val k= 5 {'acc': 0.828, 'macro_f1': 0.798} HoVer 0.668


Map: 100%|██████████| 886/886 [00:00<00:00, 889.79 examples/s]


...

val k=10 {'acc': 0.841, 'macro_f1': 0.802} HoVer 0.76


Map: 100%|██████████| 1126/1126 [00:01<00:00, 1050.38 examples/s]



test 검색된 근거 k=5 {'acc': 0.729, 'macro_f1': 0.728} HoVer 0.546
                    pred_NOT_SUPPORTED  pred_SUPPORTED
true_NOT_SUPPORTED                 383             222
true_SUPPORTED                      83             438
  근거를 다 찾은 claim  798 {'acc': 0.771, 'macro_f1': 0.768}
  근거를 놓친 claim     328 {'acc': 0.628, 'macro_f1': 0.624}


Map: 100%|██████████| 1126/1126 [00:00<00:00, 3165.28 examples/s]


test 정답 근거           {'acc': 0.783, 'macro_f1': 0.783}


In [16]:
d = dfs['test']
d['evidence_found'] = found
d[['uid', 'claim', 'label', 'pred', 'evidence_found', 'supporting_facts', 'pred_sents']] \
    .to_json(f'{PROC}/hover2_test_e2e_{TRAIN_ON}{TAG}.jsonl', orient='records', lines=True, force_ascii=False)

In [17]:
ex = dfs['train'].iloc[0]
print('claim :', ex.claim, '| label:', ex.label)
print('정답  :', ex.supporting_facts)
print('검색  :', [(t, i, round(s, 2)) for t, i, _, s in ex.pred_sents[:K_SENT]])
print('입력  :', tok.decode(train_ds[0]['input_ids']))
print('라벨  :', train_ds[0]['labels'])
print(pd.Series(train_ds['labels']).value_counts())             # 0: 약 2,300 / 1: 약 5,860 이어야 함
print(dfs['val'].pred_sents.map(len).value_counts().head())     # 대부분 10이어야 함 (빈 리스트가 많으면 문제)


claim : Skagen Painter Peder Severin Krøyer favored naturalism along with Theodor Esbern Philipsen and Kristian Zahrtmann. | label: SUPPORTED
정답  : [['Kristian Zahrtmann', 0], ['Kristian Zahrtmann', 1], ['Peder Severin Krøyer', 1]]
검색  : [('Kristian Zahrtmann', 1, 0.96), ('Theodor Philipsen', 0, 0.64), ('Kristian Zahrtmann', 0, 0.64), ('Peder Severin Krøyer', 0, 0.61), ('Skagen Painters', 3, 0.57)]
입력  : <s>Skagen Painter Peder Severin Krøyer favored naturalism along with Theodor Esbern Philipsen and Kristian Zahrtmann.</s></s>He was a part of the Danish artistic generation in the late 19th century, along with Peder Severin Krøyer and Theodor Esbern Philipsen, who broke away from both the strictures of traditional Academicism and the heritage of the Golden Age of Danish Painting, in favor of naturalism and realism. Theodor Esbern Philipsen (10 June 1840, Copenhagen - 3 March 1920, Copenhagen) was a Danish painter of Jewish ancestry; known for landscapes and animal portraits. Peder Henr

In [18]:
# 공식 HoVer score (hover-nlp.github.io 기준)
# 제출 형식: claim마다 순위가 매겨진 근거 목록 [(제목, 문장 번호), ...]
# 채점 규칙: k-hop claim은 근거 목록에서 처음 등장한 순서로 앞 k+1개 문서만, 문서마다 앞 2문장만 본다.
#           정답 문서마다 정답 문장이 최소 1개 들어 있고 라벨이 맞으면 1점.
# 공식 평가 스크립트는 공개되지 않아서(링크 404) 사이트 설명대로 구현했다. EM/F1은 이 채점 대상 문장 집합으로 계산한다.
def official_scores(d, pred_labels, n_submit):
    hover, ev, em, f1 = [], [], [], []
    for r, p in zip(d.itertuples(), pred_labels):
        per_doc, kept = {}, set()
        for t, i, *_ in r.pred_sents[:n_submit]:      # 제출할 근거 = 문장 점수 순위 상위 n_submit개
            t = nfc(t)
            if t not in per_doc:
                if len(per_doc) == r.num_hops + 1:    # 앞 k+1개 문서까지만
                    continue
                per_doc[t] = 0
            if per_doc[t] < 2:                        # 문서마다 앞 2문장까지만
                per_doc[t] += 1
                kept.add((t, int(i)))
        gold = r.gold_sents
        every_doc = all(any(g in kept for g in gold if g[0] == doc) for doc in {t for t, _ in gold})
        ev.append(every_doc)
        hover.append(every_doc and p == r.labels)
        tp = len(kept & gold)
        em.append(kept == gold)
        f1.append(2 * tp / (len(kept) + len(gold)))   # F1 = 2PR/(P+R) = 2·tp/(|예측|+|정답|)
    # evidence_ok: 근거 조건만 통과한 비율. HoVer와의 차이 = 근거는 맞았는데 라벨을 틀린 몫
    return {'HoVer': round(float(np.mean(hover)), 3), 'evidence_ok': round(float(np.mean(ev)), 3), 'evidence_EM': round(float(np.mean(em)), 3), 'evidence_F1': round(float(np.mean(f1)), 3)}

# 제출 근거 수 n은 val에서 고르고 test는 한 번만 잰다
val_pred = predict(retrieved_ds('val'))
val_res = {n: official_scores(dfs['val'], val_pred, n) for n in [2, 3, 4, 5, 10]}
for n, r in val_res.items():
    print(f'val 제출 근거 상위 {n:2d}개', r)
N_SUBMIT = max(val_res, key=lambda n: val_res[n]['HoVer'])

d = dfs['test']
print(f'\ntest 제출 근거 상위 {N_SUBMIT}개 (val로 선택)', official_scores(d, pred, N_SUBMIT))
print('참고: 우리 정의(정답 문장 전부가 top-5 안) HoVer', round(float(np.mean((pred == d.labels.values) & evidence_found(d, 5))), 3))

Map: 100%|██████████| 886/886 [00:00<00:00, 1771.42 examples/s]


val 제출 근거 상위  2개 {'HoVer': 0.56, 'evidence_ok': 0.64, 'evidence_EM': 0.468, 'evidence_F1': 0.741}
val 제출 근거 상위  3개 {'HoVer': 0.644, 'evidence_ok': 0.743, 'evidence_EM': 0.049, 'evidence_F1': 0.68}
val 제출 근거 상위  4개 {'HoVer': 0.674, 'evidence_ok': 0.779, 'evidence_EM': 0.01, 'evidence_F1': 0.645}
val 제출 근거 상위  5개 {'HoVer': 0.692, 'evidence_ok': 0.801, 'evidence_EM': 0.005, 'evidence_F1': 0.625}
val 제출 근거 상위 10개 {'HoVer': 0.7, 'evidence_ok': 0.812, 'evidence_EM': 0.0, 'evidence_F1': 0.546}

test 제출 근거 상위 10개 (val로 선택) {'HoVer': 0.579, 'evidence_ok': 0.766, 'evidence_EM': 0.0, 'evidence_F1': 0.533}
참고: 우리 정의(정답 문장 전부가 top-5 안) HoVer 0.546
